# Práctica N.° 14 — Árboles B+
## Consultas de Rango sobre el Catálogo de la Biblioteca Central UNA-PUNO — Python y C++

**Curso:** Algoritmos y Estructuras de Datos — SIS210
**Estudiante:** Francy Jimena Ramos Vilca
**Docente:** Dr. Aldo Hernán Zanabria Gálvez
**Fecha:** 27 de julio de 2026

Este notebook contiene el código completo de las Actividades 1 a 5 (Python) y el
Trabajo de Investigación (prueba de estrés y comparación de memoria), ejecutado de
forma real. Las Actividades 6 y 7 (C++17: `catalogo_bp.hpp`, `catalogo_bp.cpp`,
`benchmark_rango.cpp`) se entregan como archivos fuente separados, documentadas en el
informe (`Practica14_ArbolBMas_Ramos_Vilca_Francy_Jimena.docx`).

## Actividades 1-4: NodoHojaBP, NodoInternoBP, ArbolBMas completo

Se definen las estructuras de nodo hoja (con datos y puntero `siguiente`) e interno
(solo claves guía), y la clase `ArbolBMas` con inserción con split (Actividad 2),
búsqueda puntual y consulta de rango O(log n + k) (Actividad 3), y eliminación con
fusión/redistribución preservando la lista enlazada (Actividad 4).

**Nota de corrección importante:** la guía original de `_reparar_hoja_subpoblada` no
propaga el subpoblamiento hacia los nodos internos superiores cuando estos quedan con
menos de t-1 claves tras una fusión. Se verificó (ver celda de verificación más abajo)
que esto rompe la invariante B+ tras suficientes eliminaciones en árboles de más de 2
niveles. Esta implementación agrega `_reparar_interno_subpoblado()` para propagar la
corrección recursivamente hacia arriba, incluyendo el caso de que la raíz quede vacía.

In [1]:
# ── Actividad 1: Estructuras B+ ─────────────────────────────────────────
class NodoHojaBP:
    def __init__(self):
        self.claves = []       # codigos topograficos ordenados
        self.libros = []       # registros paralelos a claves
        self.siguiente = None  # puntero a la siguiente hoja (clave del B+)
        self.es_hoja = True


class NodoInternoBP:
    def __init__(self):
        self.claves = []  # claves GUIA (no tienen datos asociados)
        self.hijos = []    # punteros a NodoHojaBP o NodoInternoBP
        self.es_hoja = False


# ── Actividad 2: ArbolBMas — insercion ──────────────────────────────────
class ArbolBMas:
    def __init__(self, t=4):
        self.t = t
        self.raiz = NodoHojaBP()
        self.primera_hoja = self.raiz  # ancla para recorrer todo el catalogo

    def insertar(self, codigo, libro):
        resultado = self._ins(self.raiz, codigo, libro)
        if resultado:  # la raiz se dividio, crear nueva raiz interna
            clave_subida, nuevo_hermano = resultado
            nueva_raiz = NodoInternoBP()
            nueva_raiz.claves = [clave_subida]
            nueva_raiz.hijos = [self.raiz, nuevo_hermano]
            self.raiz = nueva_raiz

    def _ins(self, nodo, codigo, libro):
        if nodo.es_hoja:
            i = 0
            while i < len(nodo.claves) and codigo > nodo.claves[i]:
                i += 1
            nodo.claves.insert(i, codigo)
            nodo.libros.insert(i, libro)
            if len(nodo.claves) <= 2 * self.t - 1:
                return None
            return self._split_hoja(nodo)
        else:
            i = 0
            while i < len(nodo.claves) and codigo >= nodo.claves[i]:
                i += 1
            resultado = self._ins(nodo.hijos[i], codigo, libro)
            if not resultado:
                return None
            clave_subida, nuevo_hermano = resultado
            nodo.claves.insert(i, clave_subida)
            nodo.hijos.insert(i + 1, nuevo_hermano)
            if len(nodo.claves) <= 2 * self.t - 1:
                return None
            return self._split_interno(nodo)

    def _split_hoja(self, hoja):
        mitad = len(hoja.claves) // 2
        nueva = NodoHojaBP()
        nueva.claves = hoja.claves[mitad:]
        nueva.libros = hoja.libros[mitad:]
        hoja.claves = hoja.claves[:mitad]
        hoja.libros = hoja.libros[:mitad]
        nueva.siguiente = hoja.siguiente  # CRITICO: mantener la lista enlazada
        hoja.siguiente = nueva
        return (nueva.claves[0], nueva)  # la clave guia sube al padre

    def _split_interno(self, nodo):
        mitad = len(nodo.claves) // 2
        clave_media = nodo.claves[mitad]
        nuevo = NodoInternoBP()
        nuevo.claves = nodo.claves[mitad + 1:]
        nuevo.hijos = nodo.hijos[mitad + 1:]
        nodo.claves = nodo.claves[:mitad]
        nodo.hijos = nodo.hijos[:mitad + 1]
        return (clave_media, nuevo)

    # ── Actividad 3: Busqueda puntual y consulta de rango O(log n + k) ──
    def _bajar_a_hoja(self, codigo):
        """Desciende desde la raiz hasta la hoja donde deberia estar codigo."""
        nodo = self.raiz
        while not nodo.es_hoja:
            i = 0
            while i < len(nodo.claves) and codigo >= nodo.claves[i]:
                i += 1
            nodo = nodo.hijos[i]
        return nodo

    def buscar(self, codigo):
        hoja = self._bajar_a_hoja(codigo)
        for i, k in enumerate(hoja.claves):
            if k == codigo:
                return hoja.libros[i]
        return None

    def rango(self, codigo_min, codigo_max):
        """Retorna todos los libros con codigo en [codigo_min, codigo_max].
        Complejidad: O(log n) para llegar a la primera hoja + O(k) para
        recorrer la lista enlazada hasta superar codigo_max."""
        resultados = []
        hoja = self._bajar_a_hoja(codigo_min)
        while hoja:
            for i, k in enumerate(hoja.claves):
                if codigo_min <= k <= codigo_max:
                    resultados.append(hoja.libros[i])
                elif k > codigo_max:
                    return resultados
            hoja = hoja.siguiente  # ¡aqui esta la ventaja del B+!
        return resultados

    def recorrer_todo_el_catalogo(self):
        """Recorre TODAS las hojas en O(n) sin tocar los nodos internos."""
        resultados = []
        hoja = self.primera_hoja
        while hoja:
            resultados.extend(hoja.libros)
            hoja = hoja.siguiente
        return resultados

    # ── Actividad 4: Eliminacion B+ — preservando la lista enlazada ─────
    def eliminar(self, codigo):
        hoja = self._bajar_a_hoja(codigo)
        if codigo not in hoja.claves:
            raise KeyError(f'Codigo no encontrado: {codigo}')
        idx = hoja.claves.index(codigo)
        hoja.claves.pop(idx)
        hoja.libros.pop(idx)
        if len(hoja.claves) >= self.t - 1 or hoja is self.raiz:
            return  # la hoja sigue cumpliendo el minimo, no se requiere accion
        self._reparar_hoja_subpoblada(hoja, codigo)

    def _reparar_hoja_subpoblada(self, hoja, codigo_referencia):
        """
        Busca el padre y hermano de la hoja subpoblada. Intenta redistribuir
        (borrow) de un hermano; si no es posible, fusiona, ACTUALIZANDO
        el puntero `siguiente` para que la lista enlazada quede consistente.

        Nota de correccion (agregada en esta implementacion, no en la guia
        original): tras una fusion, el PADRE pierde una clave y un hijo; si
        el padre mismo queda con menos de t-1 claves y no es la raiz, ese
        subpoblamiento debe repararse tambien (recursivamente hacia arriba),
        igual que en el arbol B de la Practica 13. La guia original no
        contemplaba este caso -- ver seccion de correccion en el informe.
        """
        padre, idx_en_padre, hermano_izq, hermano_der = self._localizar_contexto(hoja, codigo_referencia)

        # Intentar redistribuir desde el hermano derecho
        if hermano_der and len(hermano_der.claves) > self.t - 1:
            hoja.claves.append(hermano_der.claves.pop(0))
            hoja.libros.append(hermano_der.libros.pop(0))
            padre.claves[idx_en_padre] = hermano_der.claves[0]
            return

        # Intentar redistribuir desde el hermano izquierdo
        if hermano_izq and len(hermano_izq.claves) > self.t - 1:
            hoja.claves.insert(0, hermano_izq.claves.pop())
            hoja.libros.insert(0, hermano_izq.libros.pop())
            padre.claves[idx_en_padre - 1] = hoja.claves[0]
            return

        # Sin posibilidad de redistribuir: FUSIONAR con el hermano derecho
        if hermano_der:
            hoja.claves.extend(hermano_der.claves)
            hoja.libros.extend(hermano_der.libros)
            hoja.siguiente = hermano_der.siguiente  # ¡reparar la lista!
            padre.claves.pop(idx_en_padre)
            padre.hijos.pop(idx_en_padre + 1)
        elif hermano_izq:
            hermano_izq.claves.extend(hoja.claves)
            hermano_izq.libros.extend(hoja.libros)
            hermano_izq.siguiente = hoja.siguiente  # ¡reparar la lista!
            padre.claves.pop(idx_en_padre - 1)
            padre.hijos.pop(idx_en_padre)

        # --- EXTENSION (no en la guia original): propagar el subpoblamiento
        # hacia arriba si el padre quedo con menos de t-1 claves y no es raiz.
        self._reparar_interno_subpoblado(padre)

    def _localizar_contexto(self, hoja_objetivo, codigo_ref):
        """Localiza padre, indice y hermanos de hoja_objetivo (busqueda auxiliar)."""
        camino = []
        nodo = self.raiz
        while not nodo.es_hoja:
            i = 0
            while i < len(nodo.claves) and codigo_ref >= nodo.claves[i]:
                i += 1
            camino.append((nodo, i))
            nodo = nodo.hijos[i]
        if not camino:
            return None, None, None, None
        padre, idx = camino[-1]
        herm_izq = padre.hijos[idx - 1] if idx > 0 else None
        herm_der = padre.hijos[idx + 1] if idx < len(padre.hijos) - 1 else None
        return padre, idx, herm_izq, herm_der

    # --- EXTENSION: reparacion recursiva de nodos internos subpoblados ---
    def _localizar_contexto_interno(self, nodo_objetivo):
        """Analogo a _localizar_contexto pero para un nodo INTERNO cualquiera."""
        if nodo_objetivo is self.raiz:
            return None, None, None, None
        camino = []
        nodo = self.raiz
        clave_guia = nodo_objetivo.claves[0] if nodo_objetivo.claves else None
        # Descenso guiado por identidad de objeto, no por clave (mas robusto)
        def _buscar(actual):
            if actual is nodo_objetivo:
                return True
            if actual.es_hoja:
                return False
            for hijo in actual.hijos:
                camino.append((actual, actual.hijos.index(hijo)))
                if hijo is nodo_objetivo or _buscar(hijo):
                    return True
                camino.pop()
            return False
        _buscar(self.raiz)
        if not camino:
            return None, None, None, None
        padre, idx = camino[-1]
        herm_izq = padre.hijos[idx - 1] if idx > 0 else None
        herm_der = padre.hijos[idx + 1] if idx < len(padre.hijos) - 1 else None
        return padre, idx, herm_izq, herm_der

    def _reparar_interno_subpoblado(self, nodo):
        t = self.t
        if nodo is self.raiz:
            # Si la raiz interna quedo sin claves (un solo hijo), esa hija pasa a ser la raiz
            if not nodo.es_hoja and len(nodo.claves) == 0 and len(nodo.hijos) == 1:
                self.raiz = nodo.hijos[0]
            return
        if len(nodo.claves) >= t - 1:
            return  # sigue cumpliendo el minimo, no hay nada que hacer

        padre, idx_en_padre, herm_izq, herm_der = self._localizar_contexto_interno(nodo)
        if padre is None:
            return

        # Redistribuir (borrow) desde un hermano interno con excedente
        if herm_der and len(herm_der.claves) > t - 1:
            nodo.claves.append(padre.claves[idx_en_padre])
            nodo.hijos.append(herm_der.hijos.pop(0))
            padre.claves[idx_en_padre] = herm_der.claves.pop(0)
            return
        if herm_izq and len(herm_izq.claves) > t - 1:
            nodo.claves.insert(0, padre.claves[idx_en_padre - 1])
            nodo.hijos.insert(0, herm_izq.hijos.pop())
            padre.claves[idx_en_padre - 1] = herm_izq.claves.pop()
            return

        # Fusionar con un hermano interno
        if herm_der:
            nodo.claves.append(padre.claves[idx_en_padre])
            nodo.claves.extend(herm_der.claves)
            nodo.hijos.extend(herm_der.hijos)
            padre.claves.pop(idx_en_padre)
            padre.hijos.pop(idx_en_padre + 1)
        elif herm_izq:
            herm_izq.claves.append(padre.claves[idx_en_padre - 1])
            herm_izq.claves.extend(nodo.claves)
            herm_izq.hijos.extend(nodo.hijos)
            padre.claves.pop(idx_en_padre - 1)
            padre.hijos.pop(idx_en_padre)

        # Propagar hacia arriba si el padre tambien quedo subpoblado
        self._reparar_interno_subpoblado(padre)

    # ── utilidades propias (no en la guia): altura y verificacion ──────
    def altura(self):
        n = self.raiz
        h = 0
        while not n.es_hoja:
            h += 1
            n = n.hijos[0]
        return h

    def verificar_lista_enlazada(self):
        """Verifica que la lista de hojas no tenga ciclos, cubra todas las
        hojas del arbol y este estrictamente ordenada de extremo a extremo."""
        vistos = set()
        hoja = self.primera_hoja
        claves_lista = []
        while hoja:
            if id(hoja) in vistos:
                return False, "CICLO detectado en la lista enlazada"
            vistos.add(id(hoja))
            claves_lista.extend(hoja.claves)
            hoja = hoja.siguiente
        if claves_lista != sorted(claves_lista):
            return False, "La lista enlazada no esta ordenada"
        return True, f"{len(claves_lista)} claves, sin ciclos, ordenada"


## Verificación: el algoritmo original de la guía SÍ se rompe

Se reproduce fielmente `_reparar_hoja_subpoblada` tal como aparece en la guía (sin la
propagación hacia arriba) en una subclase, y se ejecuta sobre un árbol de orden t=2 con
40 claves, eliminando en orden aleatorio, verificando la invariante B+ tras cada
eliminación.

In [2]:
"""
Prueba de verificacion: se ejecuta la logica de eliminacion TAL COMO aparece
en la guia (_reparar_hoja_subpoblada SIN la llamada final a
_reparar_interno_subpoblado) para comprobar si en efecto se rompe con un
arbol de orden pequeno y varios niveles, tal como se sospecho.
"""
import copy
from arbolbmas import ArbolBMas, NodoHojaBP, NodoInternoBP


class ArbolBMasOriginalGuia(ArbolBMas):
    """Reproduce _reparar_hoja_subpoblada exactamente como aparece en la guia,
    es decir, SIN propagar el subpoblamiento del padre hacia arriba."""

    def _reparar_hoja_subpoblada(self, hoja, codigo_referencia):
        padre, idx_en_padre, hermano_izq, hermano_der = self._localizar_contexto(hoja, codigo_referencia)

        if hermano_der and len(hermano_der.claves) > self.t - 1:
            hoja.claves.append(hermano_der.claves.pop(0))
            hoja.libros.append(hermano_der.libros.pop(0))
            padre.claves[idx_en_padre] = hermano_der.claves[0]
            return

        if hermano_izq and len(hermano_izq.claves) > self.t - 1:
            hoja.claves.insert(0, hermano_izq.claves.pop())
            hoja.libros.insert(0, hermano_izq.libros.pop())
            padre.claves[idx_en_padre - 1] = hoja.claves[0]
            return

        if hermano_der:
            hoja.claves.extend(hermano_der.claves)
            hoja.libros.extend(hermano_der.libros)
            hoja.siguiente = hermano_der.siguiente
            padre.claves.pop(idx_en_padre)
            padre.hijos.pop(idx_en_padre + 1)
        elif hermano_izq:
            hermano_izq.claves.extend(hoja.claves)
            hermano_izq.libros.extend(hoja.libros)
            hermano_izq.siguiente = hoja.siguiente
            padre.claves.pop(idx_en_padre - 1)
            padre.hijos.pop(idx_en_padre)
        # (sin propagacion hacia arriba: asi termina la funcion en la guia)


def es_arbol_valido(arbol):
    """Verifica invariantes minimas de B+: cada nodo interno no-raiz con
    >= t-1 claves, y cada hoja no-raiz con >= t-1 claves."""
    problemas = []

    def _rec(nodo, es_raiz):
        if nodo.es_hoja:
            if not es_raiz and len(nodo.claves) < arbol.t - 1:
                problemas.append(f"Hoja subpoblada: {len(nodo.claves)} claves (< t-1={arbol.t-1})")
            return
        if not es_raiz and len(nodo.claves) < arbol.t - 1:
            problemas.append(f"Nodo interno subpoblado: {len(nodo.claves)} claves (< t-1={arbol.t-1})")
        if len(nodo.hijos) != len(nodo.claves) + 1:
            problemas.append(f"Nodo interno con {len(nodo.claves)} claves pero {len(nodo.hijos)} hijos (deberia ser {len(nodo.claves)+1})")
        for h in nodo.hijos:
            _rec(h, False)

    _rec(arbol.raiz, True)
    return len(problemas) == 0, problemas


if __name__ == "__main__":
    import random
    random.seed(99)

    print("=== Verificando el algoritmo de eliminacion TAL COMO aparece en la guia ===\n")
    t = 2
    n = 40
    codigos = [f"{i:03d}" for i in range(n)]
    random.shuffle(codigos)

    arbol = ArbolBMasOriginalGuia(t=t)
    for c in codigos:
        arbol.insertar(c, {'titulo': f'Obra {c}', 'autor': 'A'})

    print(f"Arbol construido: {n} claves, t={t}, altura={arbol.altura()}")
    ok, problemas = es_arbol_valido(arbol)
    print(f"Valido tras insercion: {ok}\n")

    orden_elim = codigos.copy()
    random.shuffle(orden_elim)

    primer_fallo = None
    for i, c in enumerate(orden_elim):
        arbol.eliminar(c)
        ok, problemas = es_arbol_valido(arbol)
        if not ok and primer_fallo is None:
            primer_fallo = (i, c, problemas)

    if primer_fallo:
        i, c, problemas = primer_fallo
        print(f"CONFIRMADO: el algoritmo de la guia (sin propagacion hacia arriba) "
              f"rompe la invariante B+ en la eliminacion #{i+1} (codigo '{c}'):")
        for p in problemas:
            print(f"  - {p}")
    else:
        print("No se detectaron violaciones en esta corrida (probar con mas datos/ordenes distintos).")


=== Verificando el algoritmo de eliminacion TAL COMO aparece en la guia ===

Arbol construido: 40 claves, t=2, altura=3
Valido tras insercion: True

CONFIRMADO: el algoritmo de la guia (sin propagacion hacia arriba) rompe la invariante B+ en la eliminacion #23 (codigo '016'):
  - Nodo interno subpoblado: 0 claves (< t-1=1)


## Verificación: la versión corregida se mantiene válida en el mismo escenario

In [3]:
import random
from arbolbmas import ArbolBMas
from verificar_bug_guia import es_arbol_valido

random.seed(99)

t = 2
n = 40
codigos = [f"{i:03d}" for i in range(n)]
random.shuffle(codigos)

arbol = ArbolBMas(t=t)
for c in codigos:
    arbol.insertar(c, {'titulo': f'Obra {c}', 'autor': 'A'})

print(f"Arbol construido: {n} claves, t={t}, altura={arbol.altura()}")
ok, problemas = es_arbol_valido(arbol)
print(f"Valido tras insercion: {ok}\n")

orden_elim = codigos.copy()
random.shuffle(orden_elim)

fallos = []
for i, c in enumerate(orden_elim):
    arbol.eliminar(c)
    ok, problemas = es_arbol_valido(arbol)
    ok_lista, msg_lista = arbol.verificar_lista_enlazada()
    if not ok or not ok_lista:
        fallos.append((i, c, problemas, msg_lista))

print(f"Eliminaciones realizadas: {len(orden_elim)}")
print(f"Fallos de invariante B+ detectados: {len(fallos)}")
if fallos:
    for f in fallos[:5]:
        print(" ", f)
else:
    print("CORRECCION VERIFICADA: con la propagacion recursiva hacia arriba, "
          "el arbol se mantiene valido durante las 40 eliminaciones (mismo "
          "escenario que rompio el algoritmo original de la guia en la eliminacion #23).")

print(f"\nEstado final: {arbol.contar_claves() if hasattr(arbol, 'contar_claves') else len(arbol.recorrer_todo_el_catalogo())} libros restantes")
ok_lista, msg_lista = arbol.verificar_lista_enlazada()
print(f"Lista enlazada final: {msg_lista}")


Arbol construido: 40 claves, t=2, altura=3
Valido tras insercion: True

Eliminaciones realizadas: 40
Fallos de invariante B+ detectados: 0
CORRECCION VERIFICADA: con la propagacion recursiva hacia arriba, el arbol se mantiene valido durante las 40 eliminaciones (mismo escenario que rompio el algoritmo original de la guia en la eliminacion #23).

Estado final: 0 libros restantes
Lista enlazada final: 0 claves, sin ciclos, ordenada


## Actividad 5: Simulación — consultas de rango sobre 80,000 libros

Se indexa el mismo catálogo de 80,000 volúmenes de la Práctica 13 y se ejecutan
consultas de rango reales (sección CDU 004, y el rango amplio 000-099), verificando
además de forma cruzada contra un filtro directo sobre la lista de códigos.

In [4]:
import random
import time
random.seed(42)
catalogo_bp = ArbolBMas(t=50)
codigos = sorted(set(f'{random.randint(0,999):03d}.{random.randint(0,999):03d}'
                      for _ in range(85_000)))[:80_000]
random.shuffle(codigos)

print('Indexando 80,000 volumenes en el arbol B+...')
for i, cod in enumerate(codigos):
    catalogo_bp.insertar(cod, {'titulo': f'Obra {i}', 'autor': f'Autor {i%500}'})
print(f'Catalogo indexado. Altura del arbol B+: {catalogo_bp.altura()}')

# Consulta real: seccion 004 (Ciencias de la Computacion) completa
t0 = time.perf_counter()
resultados = catalogo_bp.rango('004.000', '004.999')
ms = (time.perf_counter() - t0) * 1000
print(f'Seccion 004 (Computacion): {len(resultados)} libros encontrados en {ms:.3f}ms')

# Consulta de un rango mas amplio: todo el rango 000-099
t0 = time.perf_counter()
resultados2 = catalogo_bp.rango('000.000', '099.999')
ms2 = (time.perf_counter() - t0) * 1000
print(f'Rango 000-099: {len(resultados2)} libros encontrados en {ms2:.3f}ms')

# Verificacion adicional (no en la guia): la lista enlazada permanece intacta
ok, msg = catalogo_bp.verificar_lista_enlazada()
print(f'\nVerificacion de la lista enlazada tras la indexacion: {msg}')

# Verificacion cruzada: comparar rango() contra un filtro directo sobre "codigos"
esperado_004 = [c for c in codigos if '004.000' <= c <= '004.999']
esperado_rango = [c for c in codigos if '000.000' <= c <= '099.999']
print(f'Verificacion cruzada seccion 004: rango()={len(resultados)} vs filtro directo={len(esperado_004)}, '
      f'coincide={len(resultados) == len(esperado_004)}')
print(f'Verificacion cruzada rango 000-099: rango()={len(resultados2)} vs filtro directo={len(esperado_rango)}, '
      f'coincide={len(resultados2) == len(esperado_rango)}')

# Busqueda puntual de control
codigo_prueba = codigos[len(codigos) // 2]
encontrado = catalogo_bp.buscar(codigo_prueba)
print(f'\nBusqueda puntual de control ({codigo_prueba}): {"encontrado" if encontrado else "NO encontrado (ERROR)"}')

Indexando 80,000 volumenes en el arbol B+...


Catalogo indexado. Altura del arbol B+: 2
Seccion 004 (Computacion): 95 libros encontrados en 0.130ms
Rango 000-099: 8095 libros encontrados en 1.399ms

Verificacion de la lista enlazada tras la indexacion: 80000 claves, sin ciclos, ordenada
Verificacion cruzada seccion 004: rango()=95 vs filtro directo=95, coincide=True
Verificacion cruzada rango 000-099: rango()=8095 vs filtro directo=8095, coincide=True

Busqueda puntual de control (209.551): encontrado


## Trabajo de Investigación (punto 2): prueba de estrés

5,000 eliminaciones aleatorias seguidas de consultas de rango repetidas sobre el
catálogo de 80,000, verificando que la lista enlazada de hojas permanece consistente
(sin ciclos, sin huecos) usando `recorrer_todo_el_catalogo()`.

In [5]:
import random
import time
# Reconstruir el mismo catalogo de 80,000 codigos de la Actividad 5
random.seed(42)
codigos = sorted(set(f'{random.randint(0,999):03d}.{random.randint(0,999):03d}'
                      for _ in range(85_000)))[:80_000]
random.shuffle(codigos)

catalogo_bp = ArbolBMas(t=50)
for i, cod in enumerate(codigos):
    catalogo_bp.insertar(cod, {'titulo': f'Obra {i}', 'autor': f'Autor {i%500}'})

print(f'Catalogo base reconstruido: {len(catalogo_bp.recorrer_todo_el_catalogo())} libros, '
      f'altura={catalogo_bp.altura()}')

# --- Prueba de estres: 5,000 eliminaciones aleatorias ---
random.seed(123)
a_eliminar = random.sample(codigos, 5_000)

for cod in a_eliminar:
    catalogo_bp.eliminar(cod)

restantes = catalogo_bp.recorrer_todo_el_catalogo()
print(f'5,000 eliminaciones completadas. Libros restantes: {len(restantes)} '
      f'(esperado: {80_000 - 5_000})')

# --- Consultas de rango repetidas tras las eliminaciones ---
rangos_prueba = [('004.000', '004.999'), ('000.000', '099.999'),
                  ('500.000', '599.999'), ('900.000', '999.999')]

print('\nConsultas de rango repetidas tras las 5,000 eliminaciones:')
codigos_restantes = set(codigos) - set(a_eliminar)
for cmin, cmax in rangos_prueba:
    t0 = time.perf_counter()
    resultado = catalogo_bp.rango(cmin, cmax)
    ms = (time.perf_counter() - t0) * 1000
    esperado = sorted(c for c in codigos_restantes if cmin <= c <= cmax)
    coincide = len(resultado) == len(esperado)
    print(f'  [{cmin}, {cmax}]: {len(resultado)} resultados en {ms:.3f}ms '
          f'(esperado: {len(esperado)}, coincide: {coincide})')

# --- Verificacion critica: lista enlazada sin ciclos, sin huecos ---
ok, msg = catalogo_bp.verificar_lista_enlazada()
print(f'\nVerificacion de la lista enlazada de hojas: {msg}')

# Verificacion cruzada final con recorrer_todo_el_catalogo()
todos = catalogo_bp.recorrer_todo_el_catalogo()
print(f'recorrer_todo_el_catalogo(): {len(todos)} libros (esperado: {len(codigos_restantes)})')

exito = ok and len(todos) == len(codigos_restantes) and len(restantes) == 80_000 - 5_000
print(f'\nPRUEBA DE ESTRES: {"EXITOSA - lista enlazada consistente tras 5,000 eliminaciones" if exito else "FALLIDA"}')

Catalogo base reconstruido: 80000 libros, altura=2
5,000 eliminaciones completadas. Libros restantes: 75000 (esperado: 75000)

Consultas de rango repetidas tras las 5,000 eliminaciones:
  [004.000, 004.999]: 85 resultados en 0.023ms (esperado: 85, coincide: True)
  [000.000, 099.999]: 7629 resultados en 0.795ms (esperado: 7629, coincide: True)
  [500.000, 599.999]: 7659 resultados en 0.808ms (esperado: 7659, coincide: True)
  [900.000, 999.999]: 6319 resultados en 0.793ms (esperado: 6319, coincide: True)

Verificacion de la lista enlazada de hojas: 75000 claves, sin ciclos, ordenada
recorrer_todo_el_catalogo(): 75000 libros (esperado: 75000)

PRUEBA DE ESTRES: EXITOSA - lista enlazada consistente tras 5,000 eliminaciones


## Trabajo de Investigación (punto 3): comparación cuantitativa de memoria

Se compara el espacio en memoria real (medido con `sys.getsizeof` de forma iterativa)
entre indexar 80,000 libros con árbol B (Práctica 13, datos posiblemente en nodos
internos) vs árbol B+ (esta práctica, datos solo en hojas), para t=50 y t=5.

In [6]:
import sys
import random
# ---------------------------------------------------------------------
# Modelo simplificado del arbol B de la Practica 13 (misma estructura de
# datos: NodoB con claves+libros paralelos en CADA nodo, incluyendo
# internos), reconstruido aqui solo para medir memoria de forma
# comparable, sin repetir toda la logica de insercion/eliminacion.
# ---------------------------------------------------------------------
class NodoB_Simulado:
    __slots__ = ('claves', 'libros', 'hijos', 'es_hoja')

    def __init__(self, es_hoja=True):
        self.claves = []
        self.libros = []  # en el arbol B, TODOS los nodos (incluyendo internos) guardan datos
        self.hijos = []
        self.es_hoja = es_hoja


def tamano_recursivo(obj_inicial):
    """Calcula el tamano total en bytes de una estructura de nodos enlazados,
    sumando sys.getsizeof de cada objeto visitado (evitando doble conteo).
    Implementado de forma ITERATIVA (pila explicita) para evitar el limite
    de recursion de Python al recorrer la lista enlazada de miles de hojas."""
    visitados = set()
    pila = [obj_inicial]
    total = 0
    while pila:
        obj = pila.pop()
        if id(obj) in visitados:
            continue
        visitados.add(id(obj))
        total += sys.getsizeof(obj)
        if isinstance(obj, dict):
            for k, v in obj.items():
                pila.append(k)
                pila.append(v)
        elif isinstance(obj, (list, tuple, set)):
            pila.extend(obj)
        elif hasattr(obj, '__slots__'):
            for slot in obj.__slots__:
                if hasattr(obj, slot):
                    pila.append(getattr(obj, slot))
        elif hasattr(obj, '__dict__'):
            pila.extend(obj.__dict__.values())
    return total


def construir_libro(i):
    # Registro representativo de un libro real (similar al usado en Practica 13/14)
    return {'titulo': f'Obra de prueba numero {i} de la coleccion UNA-PUNO',
            'autor': f'Autor Apellido {i % 500}',
            'disponible': True}


def construir_arbol_b_simulado(t, codigos):
    """Construye un arbol B (Practica 13) simplificado, insertando los
    datos DUPLICADOS en nodos internos cuando una clave asciende en un
    split, tal como ocurre en el algoritmo real de la Practica 13."""
    raiz = NodoB_Simulado(es_hoja=True)

    def split(padre, i, y):
        z = NodoB_Simulado(es_hoja=y.es_hoja)
        z.claves = y.claves[t:]
        z.libros = y.libros[t:]
        if not y.es_hoja:
            z.hijos = y.hijos[t:]
        clave_media = y.claves[t - 1]
        libro_medio = y.libros[t - 1]
        y.claves = y.claves[:t - 1]
        y.libros = y.libros[:t - 1]
        if not y.es_hoja:
            y.hijos = y.hijos[:t]
        padre.hijos.insert(i + 1, z)
        padre.claves.insert(i, clave_media)
        padre.libros.insert(i, libro_medio)

    def insertar_no_lleno(nodo, codigo, libro):
        i = len(nodo.claves) - 1
        if nodo.es_hoja:
            nodo.claves.append(None)
            nodo.libros.append(None)
            while i >= 0 and codigo < nodo.claves[i]:
                nodo.claves[i + 1] = nodo.claves[i]
                nodo.libros[i + 1] = nodo.libros[i]
                i -= 1
            nodo.claves[i + 1] = codigo
            nodo.libros[i + 1] = libro
        else:
            while i >= 0 and codigo < nodo.claves[i]:
                i -= 1
            i += 1
            if len(nodo.hijos[i].claves) == 2 * t - 1:
                split(nodo, i, nodo.hijos[i])
                if codigo > nodo.claves[i]:
                    i += 1
            insertar_no_lleno(nodo.hijos[i], codigo, libro)

    arbol = {'raiz': raiz}

    def insertar(codigo, libro):
        r = arbol['raiz']
        if len(r.claves) == 2 * t - 1:
            s = NodoB_Simulado(es_hoja=False)
            s.hijos.append(r)
            split(s, 0, r)
            arbol['raiz'] = s
        insertar_no_lleno(arbol['raiz'], codigo, libro)

    for i, cod in enumerate(codigos):
        insertar(cod, construir_libro(i))

    return arbol['raiz']


if __name__ == "__main__":
    random.seed(42)
    N = 80_000
    codigos = sorted(set(f'{random.randint(0,999):03d}.{random.randint(0,999):03d} '
                          f'{chr(65+random.randint(0,25))}{random.randint(1,99)}'
                          for _ in range(int(N * 1.06))))[:N]
    random.shuffle(codigos)

    print(f'Comparando memoria real para {len(codigos)} libros indexados...\n')

    print('Construyendo arbol B (Practica 13, t=50, datos duplicados en internos)...')
    raiz_b = construir_arbol_b_simulado(t=50, codigos=codigos)
    mem_b = tamano_recursivo(raiz_b)

    print('Construyendo arbol B+ (esta practica, t=50, datos solo en hojas)...')
    arbol_bp = ArbolBMas(t=50)
    for i, cod in enumerate(codigos):
        arbol_bp.insertar(cod, construir_libro(i))
    mem_bp = tamano_recursivo(arbol_bp.raiz)

    print(f'\nMemoria real medida (sys.getsizeof recursivo, incluye overhead de objetos Python):')
    print(f'  Arbol B  (Practica 13): {mem_b:,} bytes ({mem_b/1_048_576:.2f} MB)')
    print(f'  Arbol B+ (esta practica): {mem_bp:,} bytes ({mem_bp/1_048_576:.2f} MB)')
    print(f'  Diferencia: {mem_b - mem_bp:,} bytes ({(mem_b-mem_bp)/1_048_576:.2f} MB), '
          f'B+ usa {(1 - mem_bp/mem_b)*100:.1f}% menos memoria')

    # Contar cuantas copias de "libro" completo existen en cada estructura
    def contar_libros_en_nodos(nodo, es_hoja_attr='es_hoja'):
        total = len(nodo.libros)
        if not nodo.es_hoja:
            for h in nodo.hijos:
                total += contar_libros_en_nodos(h)
        return total

    copias_b = contar_libros_en_nodos(raiz_b)

    def contar_libros_bp(nodo):
        if nodo.es_hoja:
            return len(nodo.libros)
        return sum(contar_libros_bp(h) for h in nodo.hijos)

    copias_bp = contar_libros_bp(arbol_bp.raiz)

    print(f'\nCopias totales de registros de libro almacenadas:')
    print(f'  Arbol B:  {copias_b:,} copias para {len(codigos):,} libros unicos '
          f'({copias_b - len(codigos):,} copias duplicadas en nodos internos por splits)')
    print(f'  Arbol B+: {copias_bp:,} copias para {len(codigos):,} libros unicos '
          f'(0 duplicados: los internos solo tienen claves guia)')

    # --- Repetir la comparacion con un orden pequeno (t=5) para mostrar
    # como el ahorro de memoria escala con la proporcion de nodos internos ---
    print(f'\n=== Repitiendo la comparacion con t=5 (arbol mas "alto", mas nodos internos) ===')
    raiz_b5 = construir_arbol_b_simulado(t=5, codigos=codigos)
    mem_b5 = tamano_recursivo(raiz_b5)

    arbol_bp5 = ArbolBMas(t=5)
    for i, cod in enumerate(codigos):
        arbol_bp5.insertar(cod, construir_libro(i))
    mem_bp5 = tamano_recursivo(arbol_bp5.raiz)

    print(f'  Arbol B  (t=5): {mem_b5:,} bytes ({mem_b5/1_048_576:.2f} MB)')
    print(f'  Arbol B+ (t=5): {mem_bp5:,} bytes ({mem_bp5/1_048_576:.2f} MB)')
    print(f'  Diferencia: {mem_b5 - mem_bp5:,} bytes, B+ usa '
          f'{(1 - mem_bp5/mem_b5)*100:.1f}% menos memoria con t=5 '
          f'(frente a {(1 - mem_bp/mem_b)*100:.1f}% con t=50)')

Comparando memoria real para 80000 libros indexados...

Construyendo arbol B (Practica 13, t=50, datos duplicados en internos)...


Construyendo arbol B+ (esta practica, t=50, datos solo en hojas)...



Memoria real medida (sys.getsizeof recursivo, incluye overhead de objetos Python):
  Arbol B  (Practica 13): 32,682,839 bytes (31.17 MB)
  Arbol B+ (esta practica): 32,605,095 bytes (31.09 MB)
  Diferencia: 77,744 bytes (0.07 MB), B+ usa 0.2% menos memoria

Copias totales de registros de libro almacenadas:
  Arbol B:  80,000 copias para 80,000 libros unicos (0 copias duplicadas en nodos internos por splits)
  Arbol B+: 80,000 copias para 80,000 libros unicos (0 duplicados: los internos solo tienen claves guia)

=== Repitiendo la comparacion con t=5 (arbol mas "alto", mas nodos internos) ===


  Arbol B  (t=5): 35,917,783 bytes (34.25 MB)
  Arbol B+ (t=5): 35,518,991 bytes (33.87 MB)
  Diferencia: 398,792 bytes, B+ usa 1.1% menos memoria con t=5 (frente a 0.2% con t=50)


## Conclusión del notebook

Las Actividades 1 a 5 y el Trabajo de Investigación completo (verificación del bug de
la guía original, corrección con propagación recursiva, prueba de estrés de 5,000
eliminaciones y comparación cuantitativa de memoria) se ejecutaron realmente sobre un
catálogo de 80,000 códigos topográficos, sin errores ni pérdida de integridad en
ningún punto. El desarrollo completo en C++17 (Actividades 6 y 7, incluyendo el
benchmark real B+ vs B-Tree vs AVL), las preguntas de reflexión y la investigación
sobre PostgreSQL/InnoDB se documentan en el informe adjunto.